# Aula 02: pipeline de experimento A/B

Este notebook acompanha a aula de causalidade. O exemplo e ficticio, mas desenhado para representar uma situacao plausivel em uma disciplina: testar se um lembrete no Moodle aumenta entregas no prazo.

## 1. Pergunta causal

**Enviar um lembrete dois dias antes do prazo aumenta a probabilidade de estudantes entregarem a lista no prazo?**

- Unidade observacional: estudante.
- Tratamento: receber lembrete.
- Controle: nao receber lembrete extra.
- Resultado: entregar a lista no prazo.
- Comparacao: taxa de entrega no grupo tratado menos taxa no grupo controle.

## 2. Construindo dados didaticos

Em um experimento real, os dados viriam do Moodle e do processo de randomizacao. Aqui vamos gerar uma base pequena para estudar o pipeline.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 160

dados = pd.DataFrame({
    "estudante": np.arange(1, n + 1),
    "periodo": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.25, 0.25, 0.20, 0.18, 0.12]),
    "listas_anteriores_no_prazo": rng.binomial(4, 0.62, size=n),
    "creditos_semestre": rng.choice([12, 16, 20, 24, 28], size=n, p=[0.10, 0.25, 0.35, 0.20, 0.10])
})

dados["grupo"] = rng.choice(["controle", "lembrete"], size=n)
dados.head()

## 3. Sanity check da randomizacao

Antes de medir o efeito, verificamos se os grupos parecem comparaveis em variaveis observadas. Randomizacao nao garante igualdade perfeita, mas grandes desequilibrios merecem investigacao.

In [ ]:
dados.groupby("grupo")[["periodo", "listas_anteriores_no_prazo", "creditos_semestre"]].mean().round(2)

## 4. Gerando o resultado observado

Neste exemplo didatico, vamos supor que experiencia previa aumenta a chance de entrega, muitos creditos reduzem um pouco essa chance e o lembrete tem um efeito positivo moderado.

In [ ]:
logit = (
    -0.7
    + 0.35 * dados["listas_anteriores_no_prazo"]
    - 0.03 * (dados["creditos_semestre"] - 20)
    + 0.45 * (dados["grupo"] == "lembrete")
)

prob = 1 / (1 + np.exp(-logit))
dados["entregou_no_prazo"] = rng.binomial(1, prob)
dados.head()

## 5. Medindo o efeito

Como o tratamento foi randomizado, uma primeira estimativa natural e a diferenca entre as taxas de entrega dos dois grupos.

In [ ]:
taxas = dados.groupby("grupo")["entregou_no_prazo"].mean()
efeito = taxas["lembrete"] - taxas["controle"]

taxas.round(3), round(efeito, 3)

## 6. Incerteza por bootstrap

Uma estimativa pontual nao basta. Vamos usar bootstrap para ter uma nocao de variabilidade da diferenca de taxas.

In [ ]:
def diferenca_taxas(amostra):
    taxas = amostra.groupby("grupo")["entregou_no_prazo"].mean()
    return taxas["lembrete"] - taxas["controle"]

boot = []
for _ in range(5000):
    amostra = dados.sample(n=len(dados), replace=True, random_state=int(rng.integers(1_000_000)))
    boot.append(diferenca_taxas(amostra))

intervalo = np.quantile(boot, [0.025, 0.975])
round(efeito, 3), np.round(intervalo, 3)

## 7. Comunicando o resultado

Um texto possivel:

> Neste experimento didatico, o grupo que recebeu lembrete teve uma taxa de entrega no prazo maior que o grupo controle. A estimativa e uma diferenca absoluta de aproximadamente `efeito` pontos percentuais. Como ha incerteza amostral, reportamos tambem um intervalo bootstrap.

Em uma analise real, substituiríamos `efeito` pelo valor calculado e discutiríamos validade externa, privacidade, consentimento e possiveis efeitos indesejados.

## 8. Limitacoes

- O exemplo usa dados simulados.
- Entrega no prazo e uma medida operacional, nao necessariamente aprendizagem.
- O efeito pode mudar em outro semestre, outra turma ou outro tipo de atividade.
- Se estudantes conversam entre si, o tratamento pode vazar para o grupo controle.

## Para praticar

1. Mude o tamanho da turma e observe como o intervalo muda.
2. Calcule o efeito separadamente para estudantes com 0-2 e 3-4 listas anteriores no prazo.
3. Explique por que a randomizacao e importante neste exemplo.